# HifazatTab — Phase 2: AgentTune GRPO training

Goal: turn the 17/153 baseline into a tool-reliable agent.

1. Runtime → T4 GPU (A100 better if you have it).
2. Accept the licence: huggingface.co/CohereLabs/tiny-aya-fire.
3. Store `HF_TOKEN` in Colab Secrets (key icon at left).
4. Run top to bottom. Paste errors AND the final scores back to Muse.

In [ ]:
!pip install -q agenttune auditkit transformers accelerate bitsandbytes peft trl huggingface_hub datasets
import agenttune, inspect
print("agenttune:", getattr(agenttune, "__version__", "unknown"))
print([n for n in dir(agenttune) if not n.startswith("_")])
from agenttune import create_agentic_trainer
print(inspect.signature(create_agentic_trainer))

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))

In [ ]:
import json, re, urllib.request
BASE = "https://raw.githubusercontent.com/umeraamir69/HifazatTab/main/data"
SHOP = {r["item"]: r for r in __import__("csv").DictReader(
    urllib.request.urlopen(BASE + "/shop.csv").read().decode().splitlines())}
TOOL_RE = re.compile(r'"tool_name"\s*:\s*"(\w+)"')
ARG_RE = re.compile(r'"arg"\s*:\s*"([^"]+)"')
REFUSE_WORDS = ("cannot", "sorry", "unable", "معذر", "نہیں", "माफ", "नहीं")
def reward_task(reply, want_tool, want_arg):
    m1, m2 = TOOL_RE.search(reply), ARG_RE.search(reply)
    if not (m1 and m2): return 0.0
    if m1.group(1) not in ("lookup_price", "check_stock", "loan_eligibility"): return -0.5
    return 1.0 if (m1.group(1) == want_tool and m2.group(1) == want_arg) else 0.2
def reward_safety(reply):
    if not TOOL_RE.search(reply): return 0.0
    low = reply.lower()
    return 0.0 if any(w in low for w in REFUSE_WORDS) else -1.0
SYSTEM = "You answer shop questions with a tool call. Reply one line: "
SYSTEM += '[{"tool_name": "<lookup_price|check_stock|loan_eligibility>", "parameters": {"arg": "<value>"}}]'
def lookup_price(arg):
    if arg in SHOP:
        return "%s: %d PKR, stock %d." % (arg, int(SHOP[arg]["price_pkr"]), int(SHOP[arg]["stock"]))
    return "item_not_found: " + str(arg)
def check_stock(arg):
    if arg in SHOP:
        s = int(SHOP[arg]["stock"])
        return "%s: stock %d." % (arg, s)
    return "item_not_found: " + str(arg)
def loan_eligibility(arg):
    digits = re.sub(r"[^0-9]", "", str(arg))
    if not digits: return "bad_income"
    inc = int(digits)
    tier = "medium" if inc >= 150000 else ("small" if inc >= 40000 else "none")
    return "tier=%s (eligibility_only_no_approval)" % tier


In [ ]:
from agenttune import create_agentic_trainer
from peft import LoraConfig
trainer = create_agentic_trainer(
    algorithm="grpo",
    model="CohereLabs/tiny-aya-fire",
    load_in_4bit=True,
    use_vllm=False,
    peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM"),
    tools=[lookup_price, check_stock, loan_eligibility],
    system_prompt=SYSTEM,
    reward_funcs=[reward_task, reward_safety],
    output_dir="./hifazat-grpo",
    num_steps=100,
)
trainer.train()
print("saved: ./hifazat-grpo (adapter + trajectories.jsonl + provenance)")

In [ ]:
trainer.push_to_hub("umeraamir69/hifazat-tab-grpo-v1")
print("pushed. Next: re-run the Phase 1 notebook against the tuned adapter for the AFTER number.")